# RB-LNL-Ti: one-click train or evaluate
`RUN_MODE = 'AUTO'` trains all four stages when no final checkpoint exists, and evaluates the checkpoint when it is already available. Set `RUN_MODE = 'TRAIN'` or `RUN_MODE = 'EVALUATE'` to override.

In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')

# Leave empty when the repository was cloned before opening this notebook.
REPO_URL = ''  # e.g. https://github.com/<user>/<repo>.git
candidates = [Path.cwd(), Path('/content/rb-lnl-ti'), Path('/content/drive/MyDrive/rb-lnl-ti')]
PROJECT_ROOT = next((p for p in candidates if (p / 'upgrade_models').exists()), None)
if PROJECT_ROOT is None and REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, '/content/rb-lnl-ti'], check=True)
    PROJECT_ROOT = Path('/content/rb-lnl-ti')
if PROJECT_ROOT is None:
    raise FileNotFoundError('Clone project first or set REPO_URL in this cell.')
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
!pip install -q -r upgrade_models/requirements.txt
import torch
import timm
print('PyTorch:', torch.__version__)
print('timm:', timm.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi train/evaluate.'

In [ ]:
from pathlib import Path
from upgrade_models.train_stage import run_all_stages, evaluate_checkpoint

RUN_MODE = 'AUTO'  # AUTO, TRAIN hoặc EVALUATE
RESUME = True
CONFIG_PATH = 'upgrade_models/config_colab.yaml'
CHECKPOINT_PATH = '/content/drive/MyDrive/rb-lnl-ti/submission/rb_lnl_ti_gtsrb.pth'

if RUN_MODE == 'AUTO':
    RUN_MODE = 'EVALUATE' if Path(CHECKPOINT_PATH).exists() else 'TRAIN'
print('RUN_MODE =', RUN_MODE)

if RUN_MODE == 'TRAIN':
    results = run_all_stages(CONFIG_PATH, resume=RESUME)
    print(results)
elif RUN_MODE == 'EVALUATE':
    if not Path(CHECKPOINT_PATH).exists():
        raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')
    metrics = evaluate_checkpoint(CONFIG_PATH, CHECKPOINT_PATH)
    print(metrics)
else:
    raise ValueError('RUN_MODE must be AUTO, TRAIN or EVALUATE')

## Optional: predict new images
Set `IMAGE_PATHS` and run the next cell after training or loading a checkpoint.

In [ ]:
IMAGE_PATHS = []  # e.g. ['/content/sign.jpg']
if IMAGE_PATHS:
    import torch
    from PIL import Image
    from upgrade_models.rb_lnl_ti import RB_LNL_Ti
    from upgrade_models.pipeline import build_transforms
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RB_LNL_Ti(num_classes=43).to(device)
    state = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state, strict=True)
    model.eval()
    transform = build_transforms(False)
    with torch.no_grad():
        for image_path in IMAGE_PATHS:
            image = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
            probability = model(image).softmax(1)[0]
            score, label = probability.max(0)
            print(image_path, 'label=', int(label), 'confidence=', float(score))